# Diplomski rad - Primena transformer arhitekture neuronskih mreza za detekciju DoS napada

## Beleznica 5: Formalno poredjenje modela

Cilj ove beleznice je formalno statisticko poredjenje transformer modela i baznog modela (Random Forest), koriscenjem McNemar testa, radi utvrdjivanja da li je razlika u performansama statisticki znacajna.

### 1. Ucitavanje rezultata

In [ ]:
from pathlib import Path
import joblib
import numpy as np

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"

rf_results = joblib.load(DATA_DIR / "rf_baseline_results.pkl")
transformer_results = joblib.load(DATA_DIR / "transformer_results.pkl")

y_pred_rf = rf_results["y_pred"]
y_pred_transformer = transformer_results["y_pred"]
y_test = transformer_results["y_test"]

assert len(y_pred_rf) == len(y_test)
assert len(y_pred_transformer) == len(y_test)

print("RF tacnost:", rf_results["accuracy"])
print("Transformer tacnost:", transformer_results["accuracy"])
print("Broj test instanci:", len(y_test))

Mounted at /content/drive
RF tacnost: 0.9986922629833499
Transformer tacnost: 0.985178980477965
Broj test instanci: 116996


### 2. McNemar test

Sprovodi se McNemar test, statisticki test namenjen poredjenju dva klasifikaciona modela na istom skupu podataka, koji utvrdjuje da li je razlika u broju tacno/pogresno klasifikovanih instanci izmedju dva modela statisticki znacajna.

In [ ]:
from statsmodels.stats.contingency_tables import mcnemar

rf_correct = (y_pred_rf == y_test)
transformer_correct = (y_pred_transformer == y_test)

both_correct = np.sum(rf_correct & transformer_correct)
both_wrong = np.sum(~rf_correct & ~transformer_correct)
rf_only = np.sum(rf_correct & ~transformer_correct)
transformer_only = np.sum(~rf_correct & transformer_correct)

print(f"Oba tacna: {both_correct}")
print(f"Oba pogresna: {both_wrong}")
print(f"Samo RF tacan: {rf_only}")
print(f"Samo Transformer tacan: {transformer_only}")

table = [[both_correct, rf_only], [transformer_only, both_wrong]]
result = mcnemar(table, exact=False, correction=True)

print(f"\nMcNemar statistika: {result.statistic:.4f}")
print(f"p-vrednost: {result.pvalue:.10f}")

if result.pvalue < 0.05:
    print("\nRazlika izmedju modela je statisticki znacajna (p < 0.05)")
else:
    print("\nRazlika izmedju modela NIJE statisticki znacajna (p >= 0.05)")

Oba tacna: 115201
Oba pogresna: 92
Samo RF tacan: 1642
Samo Transformer tacan: 61

McNemar statistika: 1465.8837
p-vrednost: 0.0000000000

Razlika izmedju modela je statisticki znacajna (p < 0.05)


McNemar test pokazuje statisticki znacajnu razliku izmedju predikcija Random Forest i transformer modela (p < 0,001). Od ukupno 1.703 instance na kojima su se modeli razlikovali u pogledu tacnosti predikcije, Random Forest je bio jedini tacan na 1.642 instance, dok je transformer bio jedini tacan na 61 instanci.

Dobijena asimetrija u neslaganjima izmedju modela statisticki je znacajna prema McNemar testu. Istovremeno, apsolutna razlika u ukupnoj tacnosti iznosi 1,35 procentnih poena (99,87% za Random Forest i 98,52% za transformer model).